In [ ]:
## Set Device Here

import torch
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')

In [ ]:
## Parameter

# Dataset
DATASET = "SpiceNetlist"
DATASET_ROOT_DIRECTORY = "./data/"
DATASET_PT = DATASET_ROOT_DIRECTORY+DATASET+"_complete.pt"
DATASET_PROCESSED = DATASET+"_processed.pt"

# K-Fold
N_SPLITS = 5
MIN_NUM_EPOCHS = 5
RANDOM_STATE = 42

# Train
MAX_NUM_EPOCHS = 50
PATIENCE = 3
MIN_IMPROVEMENT = 0.0001
LEARNING_RATE = 1e-6
MAX_EPOCHS_WHERE_TEST_ACC_STUCK = 6

# Model/Plot Save
MODEL_SAVE_DIRECTORY = "./model-save"
PLOT_SAVE_DIRECTORY = "./plot"


In [ ]:
import math
from itertools import chain
import numpy as np
import torch.nn.functional as F
from scipy.sparse.csgraph import shortest_path
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, accuracy_score
from torch.nn import BCEWithLogitsLoss, Conv1d, MaxPool1d, ModuleList
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MLP, GCNConv, global_sort_pool
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import k_hop_subgraph, to_scipy_sparse_matrix, negative_sampling
from tqdm import tqdm
import GPUtil
import time
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

In [ ]:
def get_gpu_usage():
    GPUs = GPUtil.getGPUs()
    if len(GPUs) == 0:
        return "No GPU"
    gpu_info = []
    for gpu in GPUs:
        gpu_info.append(f"GPU {gpu.id}: {gpu.load * 100:.1f}% Load, {gpu.memoryUtil * 100:.1f}% Memory")
    return "; ".join(gpu_info)

In [ ]:
class MyOwnDataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None, pre_filter=None):
        super().__init__(root, transform, pre_transform, pre_filter)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def processed_file_names(self):
        return [DATASET_PROCESSED]

    def process(self):
        data_list = [torch.load(DATASET_PT)]
        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]
        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

In [ ]:
def drnl_node_labeling(edge_index, src, dst, num_nodes=None):
    src, dst = (dst, src) if src > dst else (src, dst)
    adj = to_scipy_sparse_matrix(edge_index, num_nodes=num_nodes).tocsr()
    idx = list(range(src)) + list(range(src + 1, adj.shape[0]))
    adj_wo_src = adj[idx, :][:, idx]
    idx = list(range(dst)) + list(range(dst + 1, adj.shape[0]))
    adj_wo_dst = adj[idx, :][:, idx]
    dist2src = shortest_path(adj_wo_dst, directed=False, unweighted=True, indices=src)
    dist2src = np.insert(dist2src, dst, 0, axis=0)
    dist2src = torch.from_numpy(dist2src)
    dist2dst = shortest_path(adj_wo_src, directed=False, unweighted=True, indices=dst - 1)
    dist2dst = np.insert(dist2dst, src, 0, axis=0)
    dist2dst = torch.from_numpy(dist2dst)
    dist = dist2src + dist2dst
    dist_over_2, dist_mod_2 = dist // 2, dist % 2
    z = 1 + torch.min(dist2src, dist2dst)
    z += dist_over_2 * (dist_over_2 + dist_mod_2 - 1)
    z[src] = 1.
    z[dst] = 1.
    z[torch.isnan(z)] = 0.
    return z.to(torch.long)

def extract_enclosing_subgraphs(edge_index, edge_label_index, y, num_hops, data_x, global_max_z=None):
    data_list = []
    local_max_z = 0
    for src, dst in edge_label_index.t().tolist():
        sub_nodes, sub_edge_index, mapping, _ = k_hop_subgraph(
            [src, dst], num_hops, edge_index, relabel_nodes=True, num_nodes=data_x.size(0))
        src, dst = mapping.tolist()
        mask1 = (sub_edge_index[0] != src) | (sub_edge_index[1] != dst)
        mask2 = (sub_edge_index[0] != dst) | (sub_edge_index[1] != src)
        sub_edge_index = sub_edge_index[:, mask1 & mask2]
        z = drnl_node_labeling(sub_edge_index, src, dst, num_nodes=sub_nodes.size(0))
        local_max_z = max(local_max_z, int(z.max()))
        data = Data(x=data_x[sub_nodes], z=z, edge_index=sub_edge_index, y=y)
        data_list.append(data)
    max_z = global_max_z if global_max_z is not None else local_max_z
    for data in data_list:
        one_hot = F.one_hot(data.z, max_z + 1).to(torch.float)
        data.x = torch.cat([one_hot, data.x], dim=1)
    return data_list

def generate_seal_data(data, num_hops=2):
    transform = RandomLinkSplit(num_val=0.1, num_test=0.2, is_undirected=True, split_labels=True)
    train_data, val_data, test_data = transform(data)
    train_pos_data = extract_enclosing_subgraphs(
        train_data.edge_index, train_data.pos_edge_label_index, 1, num_hops, data.x)
    train_neg_data = extract_enclosing_subgraphs(
        train_data.edge_index, train_data.neg_edge_label_index, 0, num_hops, data.x)
    val_pos_data = extract_enclosing_subgraphs(
        val_data.edge_index, val_data.pos_edge_label_index, 1, num_hops, data.x)
    val_neg_data = extract_enclosing_subgraphs(
        val_data.edge_index, val_data.neg_edge_label_index, 0, num_hops, data.x)
    test_pos_data = extract_enclosing_subgraphs(
        test_data.edge_index, test_data.pos_edge_label_index, 1, num_hops, data.x)
    test_neg_data = extract_enclosing_subgraphs(
        test_data.edge_index, test_data.neg_edge_label_index, 0, num_hops, data.x)
    return {
        'train': train_pos_data + train_neg_data,
        'val': val_pos_data + val_neg_data,
        'test': test_pos_data + test_neg_data
    }

In [ ]:
class DGCNN(torch.nn.Module):
    def __init__(self, hidden_channels, num_layers, num_features=None, k=0.6):
        super().__init__()
        if num_features is None:
            raise ValueError("num_features must be specified")
        if k < 1:
            self.k = 10
        else:
            self.k = int(k)
        self.convs = ModuleList()
        self.convs.append(GCNConv(num_features, hidden_channels))
        for i in range(0, num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, 1))
        conv1d_channels = [16, 32]
        total_latent_dim = hidden_channels * num_layers + 1
        conv1d_kws = [total_latent_dim, 5]
        self.conv1 = Conv1d(1, conv1d_channels[0], conv1d_kws[0], conv1d_kws[0])
        self.maxpool1d = MaxPool1d(2, 2)
        self.conv2 = Conv1d(conv1d_channels[0], conv1d_channels[1], conv1d_kws[1], 1)
        dense_dim = int((self.k - 2) / 2 + 1)
        dense_dim = (dense_dim - conv1d_kws[1] + 1) * conv1d_channels[1]
        self.mlp = MLP([dense_dim, 128, 1], dropout=0.5, batch_norm=False)

    def forward(self, x, edge_index, batch):
        xs = [x]
        for conv in self.convs:
            xs += [conv(xs[-1], edge_index).tanh()]
        x = torch.cat(xs[1:], dim=-1)
        x = global_sort_pool(x, batch, self.k)
        x = x.unsqueeze(1)
        x = self.conv1(x).relu()
        x = self.maxpool1d(x)
        x = self.conv2(x).relu()
        x = x.view(x.size(0), -1)
        return self.mlp(x)

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    y_pred, y_true = [], []
    with tqdm(loader, desc="Training", unit="batch", mininterval=10) as tepoch:
        for data in tepoch:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch)
            loss = criterion(out.view(-1), data.y.to(torch.float))
            loss.backward()
            optimizer.step()
            total_loss += float(loss) * data.num_graphs
            y_pred.append(out.view(-1).cpu().detach())
            y_true.append(data.y.view(-1).cpu().to(torch.float))
    train_loss = total_loss / len(loader.dataset)
    y_pred = torch.cat(y_pred)
    y_true = torch.cat(y_true)
    train_auc = roc_auc_score(y_true, y_pred)
    y_pred_binary = (torch.sigmoid(y_pred) >= 0.5).int()
    train_acc = accuracy_score(y_true, y_pred_binary)
    return train_loss, train_auc, train_acc

In [ ]:
@torch.no_grad()
def test(model, loader, mode="Validation"):
    model.eval()
    y_pred, y_true = [], []
    with tqdm(loader, desc=mode, unit="batch", mininterval=10) as tepoch:
        for data in tepoch:
            data = data.to(device)
            logits = model(data.x, data.edge_index, data.batch)
            y_pred.append(logits.view(-1).cpu())
            y_true.append(data.y.view(-1).cpu().to(torch.float))
    y_pred = torch.cat(y_pred)
    y_true = torch.cat(y_true)
    roc_auc = roc_auc_score(y_true, y_pred)
    y_pred_binary = (torch.sigmoid(y_pred) >= 0.5).int()
    accuracy = accuracy_score(y_true, y_pred_binary)
    return roc_auc, accuracy

In [ ]:
def run_kfold_experiment(n_splits=5, num_epochs=50):
    dataset = MyOwnDataset(DATASET_ROOT_DIRECTORY)
    data = dataset[0]
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    edge_indices = data.edge_index.t().cpu().numpy()
    
    val_auc_scores = []
    test_auc_scores = []
    test_acc_scores = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(edge_indices)):
        restart_fold = True
        max_restarts = 8
        restart_count = 0
        
        while restart_fold and restart_count < max_restarts:
            restart_fold = False
            if restart_count > 0:
                print(f"\n=== Restarting Fold {fold + 1}/{n_splits} (Attempt {restart_count}) ===")
            else:
                print(f"\n=== Fold {fold + 1}/{n_splits} ===")
            
            train_edges = data.edge_index[:, train_idx]
            test_edges = data.edge_index[:, test_idx]
            
            transform = RandomLinkSplit(num_val=0.1, num_test=0.0, is_undirected=True, 
                                        split_labels=True, add_negative_train_samples=True)
            train_data, val_data, _ = transform(Data(edge_index=train_edges, x=data.x))
            
            global_max_z = 0
            for edge_index, edge_label_index in [
                (train_data.edge_index, train_data.pos_edge_label_index),
                (train_data.edge_index, train_data.neg_edge_label_index),
                (val_data.edge_index, val_data.pos_edge_label_index),
                (val_data.edge_index, val_data.neg_edge_label_index),
                (test_edges, test_edges),
                (test_edges, negative_sampling(test_edges, data.x.size(0), test_edges.size(1), method="sparse"))
            ]:
                for src, dst in edge_label_index.t().tolist():
                    sub_nodes, sub_edge_index, mapping, _ = k_hop_subgraph(
                        [src, dst], 2, edge_index, relabel_nodes=True, num_nodes=data.x.size(0))
                    src, dst = mapping.tolist()
                    mask1 = (sub_edge_index[0] != src) | (sub_edge_index[1] != dst)
                    mask2 = (sub_edge_index[0] != dst) | (sub_edge_index[1] != src)
                    sub_edge_index = sub_edge_index[:, mask1 & mask2]
                    z = drnl_node_labeling(sub_edge_index, src, dst, num_nodes=sub_nodes.size(0))
                    global_max_z = max(global_max_z, int(z.max()))
            
            train_pos_data = extract_enclosing_subgraphs(
                train_data.edge_index, train_data.pos_edge_label_index, 1, 2, data.x, global_max_z)
            train_neg_data = extract_enclosing_subgraphs(
                train_data.edge_index, train_data.neg_edge_label_index, 0, 2, data.x, global_max_z)
            val_pos_data = extract_enclosing_subgraphs(
                val_data.edge_index, val_data.pos_edge_label_index, 1, 2, data.x, global_max_z)
            val_neg_data = extract_enclosing_subgraphs(
                val_data.edge_index, val_data.neg_edge_label_index, 0, 2, data.x, global_max_z)
            test_pos_data = extract_enclosing_subgraphs(train_edges, test_edges, 1, 2, data.x, global_max_z)
            neg_edge_index = negative_sampling(edge_index=train_edges, num_nodes=data.x.size(0), num_neg_samples=test_edges.size(1), method="sparse")
            test_neg_data = extract_enclosing_subgraphs(train_edges, neg_edge_index, 0, 2, data.x, global_max_z)

            train_dataset = train_pos_data + train_neg_data
            val_dataset = val_pos_data + val_neg_data
            test_dataset = test_pos_data + test_neg_data
            
            train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=1)
            test_loader = DataLoader(test_dataset, batch_size=1)
            
            num_nodes_list = sorted([d.num_nodes for d in train_dataset])
            k = num_nodes_list[int(math.ceil(0.6 * len(num_nodes_list))) - 1]
            k = max(10, k)
            num_features = train_dataset[0].x.size(1)
            model = DGCNN(hidden_channels=32, num_layers=3, num_features=num_features, k=k).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            criterion = BCEWithLogitsLoss()
            
            train_losses = []
            train_aucs = []
            train_accs = []
            val_aucs = []
            val_accs = []
            test_aucs = []
            test_accs = []
            
            best_val_auc = 0
            best_test_auc = 0
            best_test_acc = 0
            
            # Early Stopping 
            early_stop_best_val_acc = -float('inf')
            patience_counter = 0
            
            for epoch in range(1, num_epochs + 1):
                train_loss, train_auc, train_acc = train(model, train_loader, optimizer, criterion)
                val_auc, val_acc = test(model, val_loader, "Validation")
                test_auc, test_acc = test(model, test_loader, "Testing")
                
                if epoch == MAX_EPOCHS_WHERE_TEST_ACC_STUCK and abs(test_acc - 0.5) < 1e-4:
                    print(f"Test accuracy is 0.5 after 6 epochs. Restarting fold {fold}...")
                    restart_fold = True
                    restart_count += 1
                    break
                
                train_losses.append(train_loss)
                train_aucs.append(train_auc)
                train_accs.append(train_acc)
                val_aucs.append(val_auc)
                val_accs.append(val_acc)
                test_aucs.append(test_auc)
                test_accs.append(test_acc)
                
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_test_auc = test_auc
                    best_test_acc = test_acc
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'global_max_z': global_max_z,
                        'k': k,
                        'num_features': num_features
                    }, f"{MODEL_SAVE_DIRECTORY}/model_fold{fold}.pth")
                
                if val_acc > early_stop_best_val_acc + MIN_IMPROVEMENT:
                    early_stop_best_val_acc = val_acc
                    patience_counter = 0
                else:
                    if epoch > MIN_NUM_EPOCHS:
                        patience_counter += 1
                    if patience_counter > PATIENCE:
                        print(f"Early stopping triggered at epoch {epoch} for Fold {fold}")
                        break
                
                print(f"Fold {fold}, Epoch {epoch:02d}, Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}, Train Acc: {train_acc:.4f}, Val AUC: {val_auc:.4f}, Val Acc: {val_acc:.4f}, Test AUC: {test_auc:.4f}, Test Acc: {test_acc:.4f}")
            
            if restart_fold:
                continue

            ## Visualization
            # Plot 1: Training Loss
            plt.figure(figsize=(6, 5))
            plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title(f'Training Loss - Fold {fold + 1}')
            plt.legend()
            plt.tight_layout()
            plt.savefig(f'{PLOT_SAVE_DIRECTORY}/training_loss_fold{fold + 1}.png')
            plt.close()

            # Plot 2: AUC Scores
            plt.figure(figsize=(6, 5))
            plt.plot(range(1, len(train_aucs) + 1), train_aucs, label='Train AUC')
            plt.plot(range(1, len(val_aucs) + 1), val_aucs, label='Val AUC')
            plt.plot(range(1, len(test_aucs) + 1), test_aucs, label='Test AUC')
            plt.xlabel('Epoch')
            plt.ylabel('AUC Score')
            plt.title(f'AUC Scores - Fold {fold + 1}')
            plt.legend()
            plt.tight_layout()
            plt.savefig(f'{PLOT_SAVE_DIRECTORY}/auc_scores_fold{fold + 1}.png')
            plt.close()

            # Plot 3: Accuracy
            plt.figure(figsize=(6, 5))
            plt.plot(range(1, len(train_accs) + 1), train_accs, label='Train Acc')
            plt.plot(range(1, len(val_accs) + 1), val_accs, label='Val Acc')
            plt.plot(range(1, len(test_accs) + 1), test_accs, label='Test Acc')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.title(f'Accuracy - Fold {fold + 1}')
            plt.legend()
            plt.tight_layout()
            plt.savefig(f'{PLOT_SAVE_DIRECTORY}/accuracy_fold{fold + 1}.png')
            plt.close()
            
            val_auc_scores.append(best_val_auc)
            test_auc_scores.append(best_test_auc)
            test_acc_scores.append(best_test_acc)
        
        if restart_count >= max_restarts:
            print(f"Warning: Fold {fold} failed to converge after {max_restarts} restart attempts.")
            val_auc_scores.append(0.5)
            test_auc_scores.append(0.5)
            test_acc_scores.append(0.5)
    
    print("\n=== Final Results ===")
    print(f"Average Validation AUC: {np.mean(val_auc_scores):.4f} ± {np.std(val_auc_scores):.4f}")
    print(f"Average Test AUC: {np.mean(test_auc_scores):.4f} ± {np.std(test_auc_scores):.4f}")
    print(f"Average Test Accuracy: {np.mean(test_acc_scores):.4f} ± {np.std(test_acc_scores):.4f}")

In [ ]:
if __name__ == "__main__":
    run_kfold_experiment(N_SPLITS, MAX_NUM_EPOCHS)